In [0]:
%sql
CREATE TABLE Employees7 (
    emp_id INT PRIMARY KEY,
    name VARCHAR(50),
    department_id INT,
    salary INT,
    manager_id INT
);


In [0]:
%sql
CREATE TABLE Departments7 (
    dept_id INT PRIMARY KEY,
    dept_name VARCHAR(50)
);


In [0]:
%sql
CREATE TABLE Orders7 (
    order_id INT PRIMARY KEY,
    emp_id INT,
    order_amount INT,
    order_date DATE
);


In [0]:
%sql
INSERT INTO Employees7 VALUES
(1, 'Alice', 101, 60000, NULL),
(2, 'Bob', 102, 45000, 1),
(3, 'Charlie', 101, 70000, 1),
(4, 'David', 103, 40000, 2),
(5, 'Eve', 102, 50000, 2);


num_affected_rows,num_inserted_rows
5,5


In [0]:
%sql
INSERT INTO Departments7 VALUES
(101, 'HR'),
(102, 'IT'),
(103, 'Finance');


num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
INSERT INTO Orders7 VALUES
(1, 1, 5000, '2024-01-10'),
(2, 2, 3000, '2024-02-15'),
(3, 3, 7000, '2024-03-12'),
(4, 1, 2000, '2024-04-01'),
(5, 4, 1000, '2024-05-05');


num_affected_rows,num_inserted_rows
5,5


#SubQueries

In [0]:
%sql
--Find employees who earn more than average salary
select * from Employees7 where salary > ( select avg(salary) from Employees7);


emp_id,name,department_id,salary,manager_id
1,Alice,101,60000,null
3,Charlie,101,70000,1


In [0]:
%sql
--Find employees who belong to the same department as 'Alice'
select * from Employees7 where department_id = (select department_id from Employees7 where name like"Alice");

emp_id,name,department_id,salary,manager_id
1,Alice,101,60000,null
3,Charlie,101,70000,1


In [0]:
%sql
--Get employees whose salary is equal to the minimum salary
select * from Employees7 where salary = ( select min(salary) from Employees7);

emp_id,name,department_id,salary,manager_id
4,David,103,40000,2


In [0]:
%sql
--Find employees who have placed at least one order
select * from Employees7 where emp_id in (select emp_id from Orders7);

emp_id,name,department_id,salary,manager_id
1,Alice,101,60000,null
2,Bob,102,45000,1
3,Charlie,101,70000,1
4,David,103,40000,2


In [0]:
%sql
--Get employees whose salary is greater than ALL employees in department 102
select * from Employees7 where salary > (select max(salary) from Employees7 where department_id=102);

emp_id,name,department_id,salary,manager_id
1,Alice,101,60000,null
3,Charlie,101,70000,1


#CTE

In [0]:
%sql
--Create a CTE to show employees with salary > 50,000 and fetch all records
WITH cte_maxsalary as ( select * from Employees7 where salary > 50000 )
select * from cte_maxsalary;

emp_id,name,department_id,salary,manager_id
1,Alice,101,60000,null
3,Charlie,101,70000,1


In [0]:
%sql
--Find average salary per department using CTE
With cte_average as ( select avg(salary) as avg_salary,department_id from Employees7 group by department_id)
select * from cte_average;

avg_salary,department_id
65000.0,101
47500.0,102
40000.0,103


In [0]:
%sql
--Use CTE to get employees and their department names
WITH cte_joins as ( select e.name,d.dept_name from Employees7 e join Departments7 d  on e.department_id=d.dept_id )
select * from cte_joins;

name,dept_name
Alice,HR
Bob,IT
Charlie,HR
David,Finance
Eve,IT


In [0]:
%sql 
--Find total order amount per employee using CTE
WITH cte_aggregate as ( select emp_id,sum(order_amount) as Total_amount from Orders7 group by emp_id )
select * from cte_aggregate;

emp_id,sum(order_amount)
1,7000
2,3000
3,7000
4,1000


In [0]:
%sql
--Find employees whose salary is above department average using CTE
WITH Dept_Avg AS (
    SELECT department_id, AVG(salary) AS avg_salary
    FROM Employees7
    GROUP BY department_id
)
SELECT e.name, e.salary, e.department_id
FROM Employees7 e
JOIN Dept_Avg d
ON e.department_id = d.department_id
WHERE e.salary > d.avg_salary;

name,salary,department_id
Charlie,70000,101
Eve,50000,102
